# 03 - Asistente básico (RAG)

En este cuaderno construimos la primera versión del asistente: recibe una pregunta, busca en nuestros documentos los trozos que hablan de ella y le pide a un modelo de lenguaje que responda usando solo esos trozos.

## Qué es RAG

RAG (Retrieval-Augmented Generation, generación aumentada con recuperación) es una forma de usar un modelo de lenguaje para que responda con información que no conoce de serie, como la FAQ de nuestro taller o una norma concreta del BOE. Tiene tres fases:

1. **Recuperación:** convertimos la pregunta en embedding y buscamos en Chroma los trozos más parecidos (lo que ya probamos en el cuaderno 02).
2. **Aumento:** metemos esos trozos dentro del mensaje que le mandamos al modelo, junto a la pregunta. Así el modelo tiene delante la información que necesita.
3. **Generación:** el modelo redacta la respuesta en lenguaje natural a partir de esos trozos.

## Por qué hace falta un LLM

La búsqueda del cuaderno 02 solo devuelve trozos de texto, muchas veces cortados a mitad de frase y repartidos entre varios documentos. Un LLM (modelo grande de lenguaje) lee esos trozos, se queda con lo que responde a la pregunta y lo explica con una frase clara. La búsqueda encuentra la información y el LLM la redacta.

# Importaciones

- **os** y **dotenv** (load_dotenv): leen la clave de la API desde el archivo .env, para no escribirla en el código ni subirla a GitHub.
- **chromadb**: para abrir la base vectorial que creamos en el cuaderno 02.
- **SentenceTransformer**: el mismo modelo de embeddings, que necesitamos para convertir las preguntas en vectores.
- **Groq**: el cliente para hablar con el modelo de lenguaje.

In [1]:
import os
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# Clave de la API

La clave de Groq está en el archivo .env de la raíz del proyecto, que no se sube a GitHub (está en .gitignore). load_dotenv lee ese archivo y carga la clave como variable de entorno. Solo comprobamos que existe; nunca la mostramos.

In [2]:
load_dotenv('../.env')
print('Clave cargada:', os.getenv('GROQ_API_KEY') is not None)

Clave cargada: True


# Base vectorial y modelo de embeddings

Abrimos la colección taller que guardamos en ../chroma_db. No hace falta recalcular nada: los 850 trozos y sus embeddings ya están dentro.

Cargamos también intfloat/multilingual-e5-small. Es importante usar exactamente el mismo modelo que en el cuaderno 02: si no, los vectores de las preguntas y los de los trozos no serían comparables.

In [3]:
client = chromadb.PersistentClient(path='../chroma_db')
collection = client.get_collection('taller')
collection.count()

850

In [4]:
model = SentenceTransformer('intfloat/multilingual-e5-small')

# Fase 1: recuperación

Definimos una función (def) para no repetir el mismo código en cada pregunta. La función buscar:

1. Añade query: delante de la pregunta (como pide el modelo e5) y la convierte en embedding.
2. Pide a Chroma los k trozos más parecidos (por defecto 4).
3. Devuelve dos listas: los textos de los trozos y sus fuentes.

Usamos 4 trozos porque es un equilibrio: con muy pocos puede faltar la información, y con demasiados metemos ruido y el mensaje al modelo se alarga.

In [5]:
def buscar(pregunta, k=4):
    vector = model.encode('query: ' + pregunta, normalize_embeddings=True).tolist()
    resultados = collection.query(query_embeddings=[vector], n_results=k)
    trozos = resultados['documents'][0]
    fuentes = []
    for metadato in resultados['metadatas'][0]:
        fuentes.append(metadato['fuente'])
    return trozos, fuentes

Probamos la función con una pregunta y miramos de qué documentos salen los trozos.

In [6]:
trozos, fuentes = buscar('¿Cada cuánto tengo que pasar la ITV?')
fuentes

['faq_taller.md',
 'rd_920_2017_inspeccion_tecnica_vehiculos.html',
 'rd_920_2017_inspeccion_tecnica_vehiculos.html',
 'rd_920_2017_inspeccion_tecnica_vehiculos.html']

# Fase 2: aumento

## Qué es el contexto

El contexto es la información que le damos al modelo junto a la pregunta: aquí, los trozos recuperados. El modelo no sabe nada de nuestros documentos; solo puede usar lo que vaya dentro del mensaje.

La función construir_prompt numera cada trozo e indica su fuente ([1] Fuente: faq_taller.md, etc.). Así el modelo puede decir de qué documento saca cada dato, y nosotros podemos comprobarlo.

In [7]:
def construir_prompt(pregunta, trozos, fuentes):
    contexto = ''
    for i in range(len(trozos)):
        contexto = contexto + '[' + str(i + 1) + '] Fuente: ' + fuentes[i] + '\n' + trozos[i] + '\n\n'
    prompt = 'Contexto:\n' + contexto + 'Pregunta: ' + pregunta
    return prompt

In [8]:
print(construir_prompt('¿Cada cuánto tengo que pasar la ITV?', trozos, fuentes)[:600])

Contexto:
[1] Fuente: faq_taller.md
domicilio?
Sí, dentro de Valladolid capital, por 25 € ida y vuelta.

## ITV

### ¿Cada cuánto tiene que pasar la ITV un turismo?
Según el Real Decreto 920/2017, un turismo de uso particular está exento los cuatro primeros años, pasa la ITV cada dos años hasta los diez años de antigüedad y, a partir de ahí, cada año.

### ¿Qué es la pre-ITV?
Es una revisión que hacemos antes de que lleves el coche a la estación ITV: luces, frenos, neumáticos, holguras de dirección y suspensión, emisiones y fugas. Cuesta 35 € y, si encontramos defectos, te damos presupuesto pa


## El prompt de sistema

Al modelo le mandamos dos mensajes:

- **Mensaje de sistema:** las instrucciones fijas, el papel que tiene que hacer. Se repite igual en todas las preguntas.
- **Mensaje de usuario:** el contexto y la pregunta de cada vez.

En las instrucciones le pedimos dos cosas importantes:

- **Que cite la fuente:** para que el usuario pueda comprobar de dónde sale cada dato, y para que nosotros detectemos si se lo inventa.
- **Que diga no lo sé si el contexto no tiene la respuesta:** los modelos tienden a inventar respuestas que suenan convincentes (alucinaciones). En un taller una respuesta inventada sobre la ITV o una garantía es peor que reconocer que no se sabe.

In [9]:
prompt_sistema = '''Eres el asistente de un taller de vehículos en España.
Responde en español, de forma breve y clara, usando SOLO la información del contexto.
Indica al final la fuente o fuentes que has usado (el nombre del archivo).
Si el contexto no contiene la respuesta, di exactamente: No lo sé con la información disponible.'''

# Fase 3: generación

## Qué es Groq y qué modelo usamos

Groq es un servicio en la nube que ejecuta modelos de lenguaje de código abierto muy rápido y tiene un plan gratuito. Así no hace falta un ordenador potente: le mandamos el mensaje por internet y nos devuelve la respuesta.

Al listar los modelos disponibles en Groq no aparece ningún Llama de chat; ofrecen otros modelos abiertos. Elegimos **openai/gpt-oss-120b**, un modelo de pesos abiertos grande que escribe bien en español y es gratuito en Groq.

In [10]:
cliente = Groq(api_key=os.getenv('GROQ_API_KEY'))

In [11]:
modelos = []
for m in cliente.models.list().data:
    modelos.append(m.id)
sorted(modelos)

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3.8-27b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']

## Temperatura baja

La temperatura controla cuánto improvisa el modelo. Con valores altos (cerca de 1) varía más las palabras y es más creativo; con valores bajos elige siempre lo más probable. Para un asistente que debe ceñirse a los documentos nos interesa poca creatividad, así que usamos 0.1.

La función responder junta las tres fases: busca los trozos, construye el prompt, se lo manda al modelo e imprime la respuesta y las fuentes recuperadas.

In [12]:
def responder(pregunta):
    trozos, fuentes = buscar(pregunta)
    prompt = construir_prompt(pregunta, trozos, fuentes)
    respuesta = cliente.chat.completions.create(
        model='openai/gpt-oss-120b',
        temperature=0.1,
        messages=[{'role': 'system', 'content': prompt_sistema},
                  {'role': 'user', 'content': prompt}])
    print('Pregunta:', pregunta)
    print()
    print(respuesta.choices[0].message.content)
    print()
    print('Fuentes recuperadas:', fuentes)

# Ejemplos

Una pregunta por celda. Las cuatro primeras tienen respuesta en los documentos; la última no, para comprobar que dice que no lo sabe.

## Periodicidad de la ITV

In [13]:
responder('¿Cada cuánto tiene que pasar la ITV un turismo particular?')

Pregunta: ¿Cada cuánto tiene que pasar la ITV un turismo particular?

Un turismo de uso particular está exento de la ITV los primeros 4 años; después debe pasarla cada 2 años hasta que cumpla 10 años, y a partir de entonces, anualmente.  

Fuente: faq_taller.md.

Fuentes recuperadas: ['faq_taller.md', 'rd_920_2017_inspeccion_tecnica_vehiculos.html', 'rd_920_2017_inspeccion_tecnica_vehiculos.xml', 'rd_920_2017_inspeccion_tecnica_vehiculos.html']


## Llamada a revisión

In [14]:
responder('¿Qué hago si mi coche tiene una llamada a revisión?')

Pregunta: ¿Qué hago si mi coche tiene una llamada a revisión?

Si tu coche tiene una llamada a revisión, debes:

1. **Acceder al servicio de la DGT**: entra en la página de la DGT y autentícate con tu certificado o con las credenciales Cl@ve.  
2. **Consultar el aviso**: una vez identificado, selecciona el vehículo y revisa los avisos de llamadas a revisión pendientes.  
3. **Descargar la carta del fabricante**: allí encontrarás la descripción del defecto y las instrucciones precisas para solucionarlo.  
4. **Seguir las indicaciones**: normalmente deberás llevar el coche a un taller o a un servicio oficial del fabricante para que realicen la reparación o sustitución indicada, sin coste para ti.

Así garantizas que el defecto se corrija y mantienes la seguridad vial.  

**Fuente:** dgt_llamadas_a_revision.html.

Fuentes recuperadas: ['dgt_llamadas_a_revision.html', 'dgt_llamadas_a_revision.html', 'dgt_llamadas_a_revision.html', 'dgt_llamadas_a_revision.html']


## Real Decreto de talleres

In [15]:
responder('¿El taller está obligado a darme un presupuesto por escrito antes de reparar el coche?')

Pregunta: ¿El taller está obligado a darme un presupuesto por escrito antes de reparar el coche?

Sí. Según el Real Decreto 1457/1986 y la información de nuestro FAQ, el cliente tiene derecho a recibir un presupuesto por escrito, con una validez mínima de doce días hábiles, y la reparación solo se inicia cuando el cliente lo firma o renuncia por escrito a que se haga.  

**Fuentes:** faq_taller.md; rd_1457_1986_talleres_reparacion.pdf

Fuentes recuperadas: ['faq_taller.md', 'rd_1457_1986_talleres_reparacion.pdf', 'faq_taller.md', 'rd_1457_1986_talleres_reparacion.pdf']


## Estaciones ITV en Valladolid

In [16]:
responder('¿Qué estaciones ITV hay en Valladolid?')

Pregunta: ¿Qué estaciones ITV hay en Valladolid?

En el registro aparecen las siguientes estaciones ITV situadas en la provincia (y la ciudad) de Valladolid:

| Código | Operador | Dirección | Municipio |
|--------|----------|-----------|-----------|
| **ITV 4701** | GRUPO ITEVELESA, S.L. | P.I. Argales, C/. Vázquez de Menchaca, 2 | Valladolid |
| **ITV 4702** | ITEVECASA | P.I. de la Vega, Parc. 4, Tordesillas | Valladolid |
| **ITV 4703** | GRUPO ITEVELESA, S.L. | P.I. San Cristóbal, C/. Acero, Parc. 16 | Valladolid |
| **ITV 4704** | GRUPO ITEVELESA, S.L. | Polígono Industrial “Ctra. Villalpando II”, Parcelas 58‑59, Medina Rioseco | Valladolid |
| **ITV 47A2** | ITEVECASA (unidad móvil) | UNIDAD MÓVIL AGRÍCOLA Y CICLOMOTORES, Tordesillas | Valladolid |
| **ITV 47A3** | GRUPO ITEVELESA, S.L. (unidad móvil) | UNIDAD MÓVIL AGRÍCOLA Y CICLOMOTORES, Valladolid | Valladolid |

**Fuente:** registro_estaciones_itv_2026.csv.

Fuentes recuperadas: ['registro_estaciones_itv_2026.csv', 'registr

## Pregunta fuera de tema

In [17]:
responder('¿Quién ganó el Mundial de fútbol de 2010?')

Pregunta: ¿Quién ganó el Mundial de fútbol de 2010?

No lo sé con la información disponible.

Fuentes recuperadas: ['registro_estaciones_itv_2026.csv', 'registro_estaciones_itv_2026.csv', 'rd_920_2017_inspeccion_tecnica_vehiculos.html', 'registro_estaciones_itv_2026.csv']


# Observaciones

- **ITV:** respuesta correcta (exento 4 años, cada 2 años hasta los 10 y después anual), sacada de la FAQ del taller y con su fuente.
- **Llamada a revisión:** respuesta correcta y útil a partir de la página de la DGT; los 4 trozos recuperados son de esa fuente.
- **Presupuesto:** correcta (presupuesto escrito, validez mínima de doce días hábiles, firma o renuncia por escrito), citando la FAQ y el Real Decreto 1457/1986.
- **Valladolid:** lista 6 de las 7 estaciones de la provincia; falta la 4705 de Peñafiel, que no estaba en los trozos recuperados. Además los trozos del CSV mezclan estaciones de varias provincias, así que la respuesta depende de que caiga el trozo adecuado entre los 4 primeros. Es el punto débil que intentaremos mejorar con el re-ranking en el cuaderno 04.
- **Fuera de tema:** responde No lo sé con la información disponible, como pedimos en el prompt de sistema.
- Entre los trozos recuperados aparecen a la vez el HTML y el XML del Real Decreto 920/2017, que tienen el mismo contenido: ocupan huecos del contexto con información repetida.
- El modelo a veces añade detalles que no están en el contexto (por ejemplo, que la reparación de una llamada a revisión es sin coste para el usuario), por lo que conviene revisar las respuestas en la evaluación.